# AI Image Denoising — 傳統方法 vs 深度學習 比較研究
**影像處理概論 × 人工智慧與應用｜期末專案**

本 Notebook 完整涵蓋：
1. 資料準備（Set12 / BSD68 灰階、Kodak 彩色）與雜訊合成（高斯、椒鹽）
2. 傳統去噪：高斯濾波、中值濾波、雙邊濾波、Non-Local Means、BM3D
3. 深度學習去噪：DnCNN（殘差學習，盲去噪）
4. 客觀評估：PSNR / SSIM / 執行時間
5. 灰階 vs 彩色、高斯 vs 椒鹽 的比較
6. 上傳自己照片做 Demo
7. 匯出所有結果 (results.zip)

> 執行前：`Runtime → Change runtime type → T4 GPU`

## 0. 環境設定

In [ ]:
!pip -q install bm3d scikit-image

In [ ]:
import os, glob, time, math, random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import bm3d
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
torch.manual_seed(0); np.random.seed(0); random.seed(0)

## 1. 資料準備
下載灰階測試集 (Set12, BSD68) 與訓練集 (Train400)，以及彩色測試集 (Kodak)。

In [ ]:
# 灰階資料集 + 訓練影像 (來自 cszn/DnCNN)
if not os.path.exists('DnCNN'):
    !git clone --depth 1 https://github.com/cszn/DnCNN.git

def list_readable(folder_glob):
    # 找出可讀的影像，並依檔名去重 (repo 在多個資料夾放了重複的測試集)
    paths = []
    for ext in ('png', 'PNG', 'jpg', 'jpeg', 'JPG', 'bmp', 'tif', 'tiff'):
        paths += glob.glob(folder_glob + '/*.' + ext, recursive=True)
    seen = {}
    for p in sorted(paths):
        if cv2.imread(p, cv2.IMREAD_GRAYSCALE) is None:
            continue
        b = os.path.basename(p)
        if b not in seen:
            seen[b] = p
    return list(seen.values())

set12 = list_readable('DnCNN/**/Set12')
bsd68 = list_readable('DnCNN/**/Set68') + list_readable('DnCNN/**/BSD68')
train_paths = sorted(glob.glob('DnCNN/**/Train400/*.png', recursive=True))

# 後備方案：萬一測試集抓不到，從 Train400 切出測試集確保流程能跑
if len(set12) == 0:
    print('⚠️ Set12 找不到，改用 Train400 後 12 張'); set12 = train_paths[-12:]; train_paths = train_paths[:-12]
if len(bsd68) == 0:
    print('⚠️ BSD68 找不到，改用 Train400 另外 20 張'); bsd68 = train_paths[-32:-12]

print('Set12:', len(set12), '| BSD68:', len(bsd68), '| Train400:', len(train_paths))

In [ ]:
# 彩色測試影像 (Kodak)
os.makedirs('color_test', exist_ok=True)
for i in [1, 5, 7, 23]:
    fn = 'color_test/kodim%02d.png' % i
    url = 'http://r0k.us/graphics/kodak/kodak/kodim%02d.png' % i
    if not os.path.exists(fn):
        !wget -q "{url}" -O "{fn}"
color_test = sorted(glob.glob('color_test/*.png'))
print('Color test images:', len(color_test))

## 2. 雜訊合成與影像讀取
所有影像以 float32、範圍 0–255 處理，方便統一比較。

In [ ]:
def imread_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return img.astype(np.float32)

def imread_color(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32)

def add_gaussian(img, sigma):
    # 高斯雜訊 (AWGN)
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    return img + noise

def add_salt_pepper(img, amount=0.05):
    # 椒鹽雜訊
    out = img.copy()
    flat = out.reshape(-1, out.shape[-1]) if out.ndim == 3 else out.reshape(-1, 1)
    n = flat.shape[0]
    idx = np.random.choice(n, int(amount * n), replace=False)
    half = len(idx) // 2
    flat[idx[:half]] = 0
    flat[idx[half:]] = 255
    return out

## 3. 評估指標與視覺化
PSNR、SSIM 為去噪領域的標準客觀指標。

In [ ]:
def calc_psnr(a, b):
    a = np.clip(a, 0, 255); b = np.clip(b, 0, 255)
    return psnr(a, b, data_range=255)

def calc_ssim(a, b):
    a = np.clip(a, 0, 255).astype(np.float32); b = np.clip(b, 0, 255).astype(np.float32)
    if a.ndim == 3:
        return ssim(a, b, data_range=255, channel_axis=2)
    return ssim(a, b, data_range=255)

def show(images, titles, cols=None, figsize=(16, 6), save=None):
    n = len(images); cols = cols or n
    rows = math.ceil(n / cols)
    plt.figure(figsize=figsize)
    for i, (im, t) in enumerate(zip(images, titles)):
        plt.subplot(rows, cols, i + 1)
        im2 = np.clip(im, 0, 255).astype(np.uint8)
        cmap = 'gray' if im2.ndim == 2 else None
        plt.imshow(im2, cmap=cmap); plt.title(t, fontsize=9); plt.axis('off')
    plt.tight_layout()
    if save:
        os.makedirs(os.path.dirname(save), exist_ok=True)
        plt.savefig(save, dpi=150, bbox_inches='tight')
    plt.show()

## 4. 傳統去噪方法
高斯濾波、中值濾波、雙邊濾波、Non-Local Means、BM3D。輸入與輸出皆為 float32 (0–255)。

In [ ]:
def t_gaussian(noisy):
    u = np.clip(noisy, 0, 255).astype(np.uint8)
    return cv2.GaussianBlur(u, (5, 5), 0).astype(np.float32)

def t_median(noisy):
    u = np.clip(noisy, 0, 255).astype(np.uint8)
    return cv2.medianBlur(u, 3).astype(np.float32)

def t_bilateral(noisy):
    u = np.clip(noisy, 0, 255).astype(np.uint8)
    return cv2.bilateralFilter(u, 9, 75, 75).astype(np.float32)

def t_nlm(noisy, sigma=25):
    u = np.clip(noisy, 0, 255).astype(np.uint8)
    if u.ndim == 3:
        return cv2.fastNlMeansDenoisingColored(u, None, sigma, sigma, 7, 21).astype(np.float32)
    return cv2.fastNlMeansDenoising(u, None, float(sigma), 7, 21).astype(np.float32)

def t_bm3d(noisy, sigma=25):
    if noisy.ndim == 3:
        out = np.stack([bm3d.bm3d(noisy[..., c] / 255.0, sigma / 255.0) for c in range(3)], axis=2)
    else:
        out = bm3d.bm3d(noisy / 255.0, sigma / 255.0)
    return np.clip(out * 255.0, 0, 255).astype(np.float32)

## 5. 深度學習模型：DnCNN
採用殘差學習——網路預測「雜訊」，再用 `乾淨 = 輸入 - 預測雜訊`。以隨機 sigma 訓練 (盲去噪)，單一模型可處理多種雜訊強度。

In [ ]:
class DnCNN(nn.Module):
    def __init__(self, channels=1, num_layers=17, features=64):
        super().__init__()
        layers = [nn.Conv2d(channels, features, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(num_layers - 2):
            layers += [nn.Conv2d(features, features, 3, padding=1, bias=False),
                       nn.BatchNorm2d(features), nn.ReLU(inplace=True)]
        layers += [nn.Conv2d(features, channels, 3, padding=1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return x - self.net(x)   # residual learning

In [ ]:
class DenoiseDataset(Dataset):
    def __init__(self, paths, patch=40, stride=20, sigma_range=(0, 55), max_patches=60000):
        self.patches = []
        for p in paths:
            img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            h, w = img.shape
            for y in range(0, h - patch, stride):
                for x in range(0, w - patch, stride):
                    self.patches.append(img[y:y+patch, x:x+patch])
        random.shuffle(self.patches)
        self.patches = self.patches[:max_patches]
        self.sigma_range = sigma_range
        print('Total training patches:', len(self.patches))
    def __len__(self):
        return len(self.patches)
    def __getitem__(self, i):
        clean = self.patches[i].astype(np.float32)
        if random.random() < 0.5:
            clean = np.fliplr(clean).copy()
        clean = np.rot90(clean, random.randint(0, 3)).copy()
        sigma = random.uniform(*self.sigma_range)
        noisy = clean + np.random.normal(0, sigma, clean.shape).astype(np.float32)
        clean = torch.from_numpy(clean / 255.0).unsqueeze(0)
        noisy = torch.from_numpy(noisy / 255.0).unsqueeze(0)
        return noisy, clean

### 訓練
在 T4 GPU 上，25 epochs 約需 30–60 分鐘。想要更高品質可把 `EPOCHS` 調到 40–50。

In [ ]:
EPOCHS = 25
ds = DenoiseDataset(train_paths)
dl = DataLoader(ds, batch_size=128, shuffle=True, num_workers=2, drop_last=True)

model = DnCNN(channels=1).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)
crit = nn.MSELoss()

loss_hist = []
for ep in range(EPOCHS):
    model.train(); running = 0.0; t0 = time.time()
    for noisy, clean in dl:
        noisy, clean = noisy.to(device), clean.to(device)
        opt.zero_grad()
        loss = crit(model(noisy), clean)
        loss.backward(); opt.step()
        running += loss.item()
    sched.step()
    avg = running / len(dl); loss_hist.append(avg)
    print('Epoch %d/%d  loss=%.5f  time=%.1fs' % (ep + 1, EPOCHS, avg, time.time() - t0))

os.makedirs('results', exist_ok=True)
torch.save(model.state_dict(), 'results/dncnn_gray.pth')
print('Saved model -> results/dncnn_gray.pth')

In [ ]:
plt.figure(); plt.plot(range(1, len(loss_hist) + 1), loss_hist, marker='o')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.title('DnCNN Training Loss'); plt.grid(True)
plt.savefig('results/loss_curve.png', dpi=150, bbox_inches='tight'); plt.show()

### DnCNN 推論
灰階直接套用；彩色對 RGB 三通道分別處理。

In [ ]:
@torch.no_grad()
def dncnn_denoise_gray(noisy):
    model.eval()
    x = torch.from_numpy(np.clip(noisy, 0, 255).astype(np.float32) / 255.0)[None, None].to(device)
    out = model(x).clamp(0, 1)[0, 0].cpu().numpy() * 255.0
    return out

@torch.no_grad()
def dncnn_denoise_color(noisy):
    return np.stack([dncnn_denoise_gray(noisy[..., c]) for c in range(3)], axis=2)

## 6. 統一比較（灰階，高斯雜訊）
在 Set12 上，對所有方法計算平均 PSNR / SSIM / 執行時間。

In [ ]:
def evaluate_methods(images_paths, sigma, color=False):
    methods = {
        'Gaussian': t_gaussian,
        'Median': t_median,
        'Bilateral': t_bilateral,
        'NLM': lambda n: t_nlm(n, sigma),
        'BM3D': lambda n: t_bm3d(n, sigma),
        'DnCNN (ours)': dncnn_denoise_color if color else dncnn_denoise_gray,
    }
    acc = {m: [0.0, 0.0, 0.0] for m in methods}
    npsnr = 0.0; nssim = 0.0
    for path in images_paths:
        clean = imread_color(path) if color else imread_gray(path)
        noisy = add_gaussian(clean, sigma)
        npsnr += calc_psnr(clean, noisy); nssim += calc_ssim(clean, noisy)
        for m, f in methods.items():
            t0 = time.time(); out = f(noisy); dt = time.time() - t0
            acc[m][0] += calc_psnr(clean, out)
            acc[m][1] += calc_ssim(clean, out)
            acc[m][2] += dt
    N = len(images_paths)
    rows = [{'Method': 'Noisy (input)', 'PSNR': npsnr / N, 'SSIM': nssim / N, 'Time(s)': 0.0}]
    for m in methods:
        rows.append({'Method': m, 'PSNR': acc[m][0] / N, 'SSIM': acc[m][1] / N, 'Time(s)': acc[m][2] / N})
    return pd.DataFrame(rows)

In [ ]:
for sigma in [15, 25, 50]:
    print('==== Grayscale Set12, sigma = %d ====' % sigma)
    df = evaluate_methods(set12, sigma, color=False)
    print(df.to_string(index=False, float_format=lambda x: '%.3f' % x))
    df.to_csv('results/gray_set12_sigma%d.csv' % sigma, index=False)
    print()

### 灰階視覺比較圖

In [ ]:
sigma = 25
clean = imread_gray(set12[2])
noisy = add_gaussian(clean, sigma)
outs = {'Gaussian': t_gaussian(noisy), 'Median': t_median(noisy), 'Bilateral': t_bilateral(noisy),
        'NLM': t_nlm(noisy, sigma), 'BM3D': t_bm3d(noisy, sigma), 'DnCNN': dncnn_denoise_gray(noisy)}
imgs = [clean, noisy] + list(outs.values())
titles = ['Clean', 'Noisy  %.2f/%.3f' % (calc_psnr(clean, noisy), calc_ssim(clean, noisy))]
for nm, im in outs.items():
    titles.append('%s\n%.2f/%.3f' % (nm, calc_psnr(clean, im), calc_ssim(clean, im)))
show(imgs, titles, cols=4, figsize=(16, 8), save='results/visual_gray.png')

## 7. 彩色實驗（Kodak，高斯雜訊 sigma=25）

In [ ]:
print('==== Color Kodak, sigma = 25 ====')
df_c = evaluate_methods(color_test, 25, color=True)
print(df_c.to_string(index=False, float_format=lambda x: '%.3f' % x))
df_c.to_csv('results/color_kodak_sigma25.csv', index=False)

In [ ]:
sigma = 25
clean = imread_color(color_test[0])
noisy = add_gaussian(clean, sigma)
outs = {'Gaussian': t_gaussian(noisy), 'Median': t_median(noisy), 'Bilateral': t_bilateral(noisy),
        'NLM': t_nlm(noisy, sigma), 'BM3D': t_bm3d(noisy, sigma), 'DnCNN': dncnn_denoise_color(noisy)}
imgs = [clean, noisy] + list(outs.values())
titles = ['Clean', 'Noisy  %.2f/%.3f' % (calc_psnr(clean, noisy), calc_ssim(clean, noisy))]
for nm, im in outs.items():
    titles.append('%s\n%.2f/%.3f' % (nm, calc_psnr(clean, im), calc_ssim(clean, im)))
show(imgs, titles, cols=4, figsize=(18, 9), save='results/visual_color.png')

## 8. 椒鹽雜訊實驗
預期：中值濾波對椒鹽雜訊最有效；DnCNN（只訓練過高斯）表現變差——這是很好的討論點。

In [ ]:
clean = imread_gray(set12[2])
noisy = add_salt_pepper(clean, 0.05)
methods = {'Gaussian': t_gaussian, 'Median': t_median, 'Bilateral': t_bilateral,
           'NLM': lambda n: t_nlm(n, 25), 'BM3D': lambda n: t_bm3d(n, 25), 'DnCNN': dncnn_denoise_gray}
imgs = [clean, noisy]; titles = ['Clean', 'Noisy  %.2f/%.3f' % (calc_psnr(clean, noisy), calc_ssim(clean, noisy))]
rows = []
for nm, f in methods.items():
    out = f(noisy); imgs.append(out)
    p, s = calc_psnr(clean, out), calc_ssim(clean, out)
    titles.append('%s\n%.2f/%.3f' % (nm, p, s)); rows.append({'Method': nm, 'PSNR': p, 'SSIM': s})
show(imgs, titles, cols=4, figsize=(16, 8), save='results/saltpepper.png')
pd.DataFrame(rows).to_csv('results/saltpepper_set12img3.csv', index=False)

## 9. 上傳自己的照片做 Demo
情境 A：上傳乾淨照片 → 程式加雜訊 → 去噪（可算 PSNR）。

In [ ]:
from google.colab import files
up = files.upload()
for fn in up:
    img = imread_color(fn)
    noisy = add_gaussian(img, 25)
    out_bm3d = t_bm3d(noisy, 25); out_dncnn = dncnn_denoise_color(noisy)
    show([img, noisy, out_bm3d, out_dncnn],
         ['Original', 'Noisy  %.2f' % calc_psnr(img, noisy),
          'BM3D  %.2f' % calc_psnr(img, out_bm3d),
          'DnCNN  %.2f' % calc_psnr(img, out_dncnn)],
         cols=4, figsize=(18, 6), save='results/demo_' + os.path.splitext(fn)[0] + '.png')

## 10. 匯出所有結果
下載 `results.zip`（含模型權重、CSV 表格、所有圖片），方便放進報告與 GitHub。

In [ ]:
!zip -qr results.zip results
from google.colab import files as gfiles
gfiles.download('results.zip')